# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### Setup
Installs dependencies, defines helpers and (for Notebooks 3-7) loads the shared `config.json` written by Notebook 2. Run this cell first.

In [4]:
# ---- Environment setup (Google Colab / local Jupyter) ----
import os, sys, glob, zipfile, json, random, warnings
warnings.filterwarnings("ignore")

USE_DRIVE = False   # set True on Colab to keep artifacts (chroma_db, adapters, csv) across runtime resets
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/RAG/Dataset"
    os.makedirs(WORKDIR, exist_ok=True)
    os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

%pip install -q transformers accelerate bitsandbytes sentence-transformers chromadb langchain langchain-community langchain-huggingface langchain-chroma

def find_sop_dir():
    """Locate the folder holding the Markdown SOP files (corporate_policies/ or Dataset/sop_documents/).
    If it is not present, try to unzip a Dataset*.zip, and finally ask for an upload on Colab."""
    def _search():
        for c in ["corporate_policies", os.path.join("Dataset", "sop_documents"), "sop_documents"]:
            if os.path.isdir(c) and glob.glob(os.path.join(c, "*.md")):
                return c
        for pat in ["**/sop_documents", "**/corporate_policies"]:
            for h in glob.glob(pat, recursive=True):
                if glob.glob(os.path.join(h, "*.md")):
                    return h
        return None
    found = _search()
    if found: return found
    for z in glob.glob("*.zip") + glob.glob("**/Dataset*.zip", recursive=True):
        with zipfile.ZipFile(z) as zf: zf.extractall(".")
        found = _search()
        if found: return found
    try:
        from google.colab import files
        print("SOP folder not found - please upload Dataset.zip (contains sop_documents/*.md)")
        for name in files.upload():
            if name.endswith(".zip"):
                with zipfile.ZipFile(name) as zf: zf.extractall(".")
        found = _search()
        if found: return found
    except ImportError:
        pass
    raise FileNotFoundError("Could not find the SOP .md files. Place them in ./corporate_policies/ or ./Dataset/sop_documents/")

# ---- Load the single-source-of-truth configuration written by Notebook 2 ----
if not os.path.exists("/content/drive/MyDrive/RAG/Dataset/Output/config.json"):
    raise FileNotFoundError("config.json not found - run Notebook 2 (Data Preparation) first.")
cfg = json.load(open("/content/drive/MyDrive/RAG/Dataset/Output/config.json"))
MODEL_ID       = cfg["MODEL_ID"]
EMBED_MODEL    = cfg["EMBED_MODEL"]
SEED           = cfg["SEED"]
BASELINE_SYSTEM = cfg["BASELINE_SYSTEM"]
RAG_SYSTEM     = cfg["RAG_SYSTEM"]          # contains the {context} placeholder
ROUTER_SYSTEM  = cfg["ROUTER_SYSTEM"]
MAX_NEW_TOKENS = cfg["MAX_NEW_TOKENS"]
CHROMA_DIR     = cfg["CHROMA_DIR"]
COLLECTION     = cfg["COLLECTION"]
random.seed(SEED)
print("MODEL_ID:", MODEL_ID, "| EMBED_MODEL:", EMBED_MODEL, "| SEED:", SEED)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HAS_CUDA = torch.cuda.is_available()
COMPUTE_DTYPE = torch.bfloat16 if (HAS_CUDA and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
print("CUDA available:", HAS_CUDA, "| compute dtype:", COMPUTE_DTYPE)
if not HAS_CUDA:
    print("WARNING: no GPU found. Runtime -> Change runtime type -> T4 GPU. Falling back to (slow) CPU float32.")

def load_base_model(model_id=None, quantize=True):
    """Load tokenizer + base model. 4-bit NF4 quantisation on GPU (about 1 GB VRAM for the 1.5B model)."""
    model_id = model_id or MODEL_ID
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"                       # required for batched decoder-only generation
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if HAS_CUDA and quantize:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE)
        model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map={"": 0})
    elif HAS_CUDA:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=COMPUTE_DTYPE, device_map={"": 0})
    else:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
    model.eval()
    return tok, model

@torch.no_grad()
def generate_chat(model, tok, messages_list, max_new_tokens=None, batch_size=16):
    """Deterministic (greedy) batched generation. do_sample=False with temperature/top_p/top_k=None
    (never temperature=0.0 together with do_sample=False). Returns only the newly generated text."""
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
    outs = []
    for i in range(0, len(messages_list), batch_size):
        batch = messages_list[i:i + batch_size]
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
        new_tokens = gen[:, enc["input_ids"].shape[1]:]
        outs += [t.strip() for t in tok.batch_decode(new_tokens, skip_special_tokens=True)]
    return outs

def baseline_messages(query):
    return [{"role": "system", "content": BASELINE_SYSTEM}, {"role": "user", "content": query}]

def rag_messages(query, context):
    return [{"role": "system", "content": RAG_SYSTEM.format(context=context)}, {"role": "user", "content": query}]

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings
try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma

def get_embeddings():
    # SAME model + settings in every notebook (NB4, NB5, NB7). CPU is enough for MiniLM.
    return HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": "cpu"},
                                 encode_kwargs={"normalize_embeddings": True})

def load_vector_db(embeddings):
    return Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings, collection_name=COLLECTION)

Working directory: /content
MODEL_ID: Qwen/Qwen2.5-1.5B-Instruct | EMBED_MODEL: sentence-transformers/all-MiniLM-L6-v2 | SEED: 42
CUDA available: True | compute dtype: torch.float16


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [5]:
import glob, shutil
import numpy as np, pandas as pd
from langchain_community.document_loaders import TextLoader

# --- load SOP documents (one Document per .md file) ---------------------------------------------------
SOP_DIR = find_sop_dir()
docs = []
for f in sorted(glob.glob(os.path.join(SOP_DIR, "*.md"))):
    d = TextLoader(f, encoding="utf-8").load()[0]
    d.metadata["doc_name"] = os.path.basename(f)
    d.metadata["title"] = d.page_content.splitlines()[0].lstrip("# ").strip()
    docs.append(d)
print(f"Loaded {len(docs)} SOP documents:", [d.metadata["doc_name"] for d in docs])

# --- initialise the embedding model (CPU, 384-dim, normalised vectors) ---------------------------------
embeddings = get_embeddings()
sample_vec = embeddings.embed_query("Where is my refund?")
print("Sample embedding length:", len(sample_vec), "(expected 384 for all-MiniLM-L6-v2)")
assert len(sample_vec) == 384

# --- embed + validate the SOP documents ----------------------------------------------------------------
doc_vectors = np.array(embeddings.embed_documents([d.page_content for d in docs]))
print("Document embedding matrix:", doc_vectors.shape)
assert doc_vectors.shape == (len(docs), 384) and not np.isnan(doc_vectors).any()
print("Vector norms (should be ~1.0 because vectors are normalised):", np.round(np.linalg.norm(doc_vectors, axis=1)[:5], 3))
np.save("sop_embeddings.npy", doc_vectors)     # embedding storage

# semantic sanity check: similar meaning -> close coordinates
q = np.array(embeddings.embed_query("I want my money back"))
sims = doc_vectors @ q
print("\nSanity check - 'I want my money back' is closest to:")
for i in np.argsort(-sims)[:3]:
    print(f"   {sims[i]:.3f}  {docs[i].metadata['doc_name']}")

Loaded 13 SOP documents: ['account_recovery.md', 'billing_disputes.md', 'data_privacy.md', 'escalation_matrix.md', 'order_tracking.md', 'password_reset.md', 'payment_methods.md', 'product_return.md', 'refund_policy.md', 'shipping_delays.md', 'subscription_cancellation.md', 'technical_troubleshooting.md', 'working_hours.md']


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sample embedding length: 384 (expected 384 for all-MiniLM-L6-v2)
Document embedding matrix: (13, 384)
Vector norms (should be ~1.0 because vectors are normalised): [1. 1. 1. 1. 1.]

Sanity check - 'I want my money back' is closest to:
   0.348  product_return.md
   0.328  refund_policy.md
   0.282  billing_disputes.md


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [6]:
# --- build a persistent Chroma index (rebuild cleanly so re-runs do not duplicate documents) --------------
shutil.rmtree(CHROMA_DIR, ignore_errors=True)
vector_db = Chroma.from_documents(docs, embeddings, persist_directory=CHROMA_DIR, collection_name=COLLECTION,
                                  collection_metadata={"hnsw:space": "cosine"})
count = vector_db._collection.count()
print(f"Chroma index built at {CHROMA_DIR} | documents indexed: {count} | SOP files: {len(docs)}")
assert count == len(docs), "index count must equal the number of SOP documents"

# --- similarity-search configuration ---------------------------------------------------------------------
RETRIEVAL_K = cfg["RETRIEVAL_K"]            # k=1 (focused). k=3 if SOPs overlap, k=5 max (long prompts)
retriever = vector_db.as_retriever(search_kwargs={"k": RETRIEVAL_K})

# --- validate retrieval -------------------------------------------------------------------------------
hit = vector_db.similarity_search("refund", k=1)[0]
print("similarity_search('refund', k=1) ->", hit.metadata["doc_name"])
print("   retrieval sanity check:", "PASS" if hit.metadata["doc_name"] == "refund_policy.md" else "CHECK - unexpected document")
for qq in ["How do I reset my password?", "talk to a human", "when is support open"]:
    h = vector_db.similarity_search_with_score(qq, k=1)[0]
    print(f"   {qq!r:35} -> {h[0].metadata['doc_name']}  (distance {h[1]:.3f})")

# --- persistence check: re-open from disk with the same embedding model -------------------------------------
reloaded = load_vector_db(embeddings)
print("Reloaded from disk, count =", reloaded._collection.count())
assert reloaded._collection.count() == len(docs)
print("Learner inference: the index finds documents by MEANING - it returns the SOP whose coordinates are closest to the query.")

Chroma index built at ./chroma_db | documents indexed: 13 | SOP files: 13
similarity_search('refund', k=1) -> refund_policy.md
   retrieval sanity check: PASS
   'How do I reset my password?'       -> password_reset.md  (distance 0.446)
   'talk to a human'                   -> working_hours.md  (distance 0.877)
   'when is support open'              -> working_hours.md  (distance 0.567)
Reloaded from disk, count = 13
Learner inference: the index finds documents by MEANING - it returns the SOP whose coordinates are closest to the query.


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [8]:
import time, re
outputs = json.load(open("/content/drive/MyDrive/RAG/Dataset/Output/outputs.json"))
test_query, ground_truth = outputs["test_query"], outputs["ground_truth"]

tokenizer, base_model = load_base_model(MODEL_ID, quantize=True)

# --- Naive RAG: the RAW customer query goes straight into the vector DB ---------------------------------------
top = vector_db.similarity_search_with_score(test_query, k=3)
print("Top-3 retrieval for the raw query (k=3 shown for analysis; the pipeline uses k=1):")
for d, s in top:
    print(f"   distance {s:.3f}  {d.metadata['doc_name']}")
retrieved = vector_db.similarity_search(test_query, k=RETRIEVAL_K)[0]
retrieved_source = retrieved.metadata["doc_name"]
expected_source = "shipping_delays.md"
print(f"\nRetrieved (k=1): {retrieved_source} | expected SOP: {expected_source}")
if retrieved_source != expected_source:
    print("-> Naive RAG retrieved the WRONG policy for this ambiguous query - the weakness the fine-tuned router fixes.")
else:
    print("-> Correct SOP retrieved for this query (naive retrieval can still fail on other noisy queries - see Notebook 5).")

# --- prompt augmentation: inject the SOP through the system prompt ----------------------------------------------
messages = rag_messages(test_query, retrieved.page_content)
print("\nSystem prompt preview:\n" + messages[0]["content"][:300] + " ...\n")
naive_rag_output = generate_chat(base_model, tokenizer, [messages], batch_size=1)[0]
print("NAIVE RAG OUTPUT:\n" + "-" * 70 + "\n" + naive_rag_output + "\n" + "-" * 70)
print("\nBaseline output for comparison:\n" + outputs["baseline_output"])
print(f"\nGround-truth rule: {ground_truth}")
print("Mentions 3-7 business days:", bool(re.search(r"\b3\s*(?:-|to|–)\s*7\b", naive_rag_output)))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Top-3 retrieval for the raw query (k=3 shown for analysis; the pipeline uses k=1):
   distance 0.285  order_tracking.md
   distance 0.302  shipping_delays.md
   distance 0.642  refund_policy.md

Retrieved (k=1): order_tracking.md | expected SOP: shipping_delays.md
-> Naive RAG retrieved the WRONG policy for this ambiguous query - the weakness the fine-tuned router fixes.

System prompt preview:
You are a customer support assistant. Answer strictly using this SOP:
# Order Tracking

## Scope
Helps customers locate an order and interpret its status. Orders that have
clearly exceeded their delivery window follow the shipping delays procedure.

## Order States
- **Processing:** payment confirme ...

NAIVE RAG OUTPUT:
----------------------------------------------------------------------
The standard delivery times vary depending on your location but generally range from 5 to 7 business days. Please ensure that there were no mistakes during the ordering process such as incorrect addresses or

---
## Save Artifacts for Downstream Notebooks

In [10]:
import re
outputs["naive_rag_output"] = naive_rag_output
outputs["naive_rag_retrieved_source"] = retrieved_source
outputs["naive_rag_retrieval_correct"] = retrieved_source == "shipping_delays.md"
json.dump(outputs, open("outputs.json", "w"), indent=2)

assert os.path.isdir(CHROMA_DIR) and os.listdir(CHROMA_DIR), "chroma_db missing"
print("outputs.json keys:", list(json.load(open("outputs.json")).keys()))
print("./chroma_db contents:", os.listdir(CHROMA_DIR))

outputs.json keys: ['test_query', 'ground_truth', 'baseline_output', 'baseline_assessment', 'model_id', 'naive_rag_output', 'naive_rag_retrieved_source', 'naive_rag_retrieval_correct']
./chroma_db contents: ['ee2e6cc6-b1f2-475d-99a4-ad7b0b4a7d0d', 'chroma.sqlite3']


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**